# Step 36 — Car layer against road traffic counts: cordon screenlines

**Question.** The 2022 car layer (`Output/ths2017/three_mode_2022/car_2022_taz.csv`, residents'
person trips by car, both ends in the study area, 06:00–09:00) has never been checked against a
count. The Emme network delivered on 23 September 2026 (`Input/Network_with_Counts/`) carries
hourly counts in passenger-car equivalents (PCE; `YARAM6` … `YARAM19`, one column per hour of the
day) on 3,241 directional links at 698 count stations, dated 2017–2023.

**Method — cordons, no assignment.** A closed cordon around a set of TAZs is crossed exactly once
by every trip with one end inside and one outside, whatever route it takes, and twice (in, then
out) by a through trip, so the OD matrix can be compared with counts without a network
assignment: the demand crossing the cordon — person trips with exactly one end inside, by
direction, plus the through trips (both ends outside, straight centroid-to-centroid line passing
through the cordon polygon; one crossing in each direction) — is converted to vehicles with the
survey's own AM car occupancy, and the counts on every network link that crosses the cordon
boundary are summed by direction. Links that cross the boundary without a count are given the median PCE per lane of
the counted links of the same road type in the same cordon (flagged; the counted-only sum is
reported beside the imputed one, and the ratio is given against both — the counted-only ratio is
an upper bound, as if the uncounted crossings carried nothing). PCE are converted to vehicles with
an assumed factor. Five
cordons on the superzone geography: Haifa city, the Krayot, Kiryat Ata, Nazareth, and the
metropolitan core (Haifa + Nesher + the Krayot + Kiryat Ata). The counts' hourly profile on the
same links gives an independent car peak-hour factor to set against the survey's departure-time
factor of step 20.

**What the ratio should be.** Less than one: the counts hold trucks and vans, taxis, buses,
non-residents, trips with one end outside the study area, and trips of residents the survey does
not cover; the survey layer holds residents' personal car trips only. The expected order is
0.7–0.9 on an urban cordon in the AM peak; the value and its stability across cordons and
directions are the test, and the directional asymmetry (Haifa-bound in the morning) is the second.

In [1]:
import os, sys, json, time, warnings
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from shapely.geometry import Point
warnings.filterwarnings('ignore')
while not os.path.exists('METHODOLOGY.md'): os.chdir('..')
OUT = 'Output/validation'; os.makedirs(OUT, exist_ok=True); os.makedirs('Output/figures', exist_ok=True)
BLUE, ORANGE, AQUA, PURPLE, INK, MUTED, GRID = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6', '#0b0b0b', '#898781', '#e1e0d9'
CRS = 2039  # Israel TM Grid
t0 = time.time()

NET = 'Input/Network_with_Counts/Emme_Links_Final_Res 2026-09-23.shp'
net = gpd.read_file(NET, encoding='utf-8').to_crs(CRS)
HOURS = [f'YARAM{h}' for h in range(6, 20)]
net['pce_6_9'] = net[['YARAM6', 'YARAM7', 'YARAM8']].sum(axis=1)
net['counted'] = net['pce_6_9'] > 0
# count year: DATE is dd/mm/yyyy or an Excel serial; F2023COUNT marks a 2023 count file
num = pd.to_numeric(net['DATE'], errors='coerce')
yr = pd.to_datetime(net['DATE'].where(num.isna()), format='%d/%m/%Y', errors='coerce').dt.year
yr.loc[num.notna()] = pd.to_datetime(num.dropna(), unit='D', origin='1899-12-30').dt.year
yr = yr.where(yr.notna(), np.where(net['F2023COUNT'].notna(), 2023, np.nan))
net['count_year'] = yr
net['auto'] = net['MODES'].fillna('').str.contains('a') & (net['TYPE'] != 9)   # car links; TYPE 9 = centroid connectors
print(f"network: {len(net):,} directional links; counted {net.counted.sum():,} at {net.loc[net.counted, 'ID_COUNT'].nunique()} stations; "
      f"car links {net.auto.sum():,}  [{time.time()-t0:.0f} s]")
print('count year of the counted links:', net.loc[net.counted, 'count_year'].value_counts(dropna=False).sort_index().to_dict())

taz = gpd.read_file('Input/TAZ_North/TAZ_North.shp').to_crs(CRS)
taz['TAZ'] = taz['TAZ_NUMBER'].astype(int); taz['SZ'] = taz['SUPERZONE'].astype(int)
sz_names = pd.read_csv('Input/sz_localities.csv').set_index('SZ_NEW')['main localities']
car = pd.read_csv('Output/ths2017/three_mode_2022/car_2022_taz.csv', index_col=0); car.index = car.index.astype(int); car.columns = car.columns.astype(int)
taxi = pd.read_csv('Output/ths2017/three_mode_2022/taxi_2022_taz.csv', index_col=0); taxi.index = taxi.index.astype(int); taxi.columns = taxi.columns.astype(int)
bus = pd.read_csv('Output/ths2017/three_mode_2022/bus_2022_taz.csv', index_col=0); bus.index = bus.index.astype(int); bus.columns = bus.columns.astype(int)
TAZ = list(car.index); print(f"car layer 2022: {car.values.sum():,.0f} person trips on {len(TAZ)} TAZs; polygons {len(taz)}, matched {taz.TAZ.isin(TAZ).sum()}")

network: 29,710 directional links; counted 3,241 at 698 stations; car links 25,340  [1 s]
count year of the counted links: {2017.0: 1085, 2018.0: 211, 2019.0: 66, 2020.0: 324, 2021.0: 921, 2022.0: 156, 2023.0: 271, nan: 207}


car layer 2022: 1,353,798 person trips on 778 TAZs; polygons 781, matched 778


## Conversion factors

**Car occupancy** from the survey itself, so that the person-trip layer and the vehicle count are on
the same footing: the car layer holds driver (code 10) and passenger (code 11) trips, and a vehicle
is a driver trip, so vehicles = person trips × (driver trips ÷ (driver + passenger trips)). On the
AM trips (departure 06:00–08:59, `new_wf` weights) the ratio is 1.33 persons per car (all day 1.46);
the `numAccomp` field gives 1.67 because it counts companions who are not surveyed persons. 1.33 is
used, with 1.25–1.45 as the range. **PCE per vehicle**: the counts are in passenger-car equivalents
and the heavy-vehicle share is not in the file; 1.10 is assumed (range 1.05–1.20), the usual order
for an urban arterial AM peak with 5–8 % heavy vehicles at 2 PCE each. **Other survey layers on the
road**: taxi-type person trips at 1.5 per vehicle, bus person trips at 20 per bus and 2.5 PCE per
bus — small, shown separately, and not in the headline ratio.

In [2]:
OCC, OCC_RANGE = 1.33, (1.25, 1.45)
PCE, PCE_RANGE = 1.10, (1.05, 1.20)
TAXI_OCC, BUS_LOAD, BUS_PCE = 1.5, 20.0, 2.5
try:
    trips = pd.read_excel('Input/THS_2017-2018/trips_ths_2017.xlsx', engine='openpyxl')
    am = trips[trips.Dep_h.between(6, 8)]
    drv, pas = am[am.mainmode == 10]['new_wf'].sum(), am[am.mainmode == 11]['new_wf'].sum()
    occ_survey = (drv + pas) / drv
    print(f"survey AM car occupancy (driver + passenger) / driver = {occ_survey:.3f} (driver trips {drv:,.0f}, passenger {pas:,.0f}); used {OCC}")
except Exception as e:
    print('trips file not readable here (LFS pointer?) — occupancy 1.33 taken from the run of 23 September 2026:', str(e)[:60])

survey AM car occupancy (driver + passenger) / driver = 1.332 (driver trips 1,223,832, passenger 406,306); used 1.33


## Cordons

Superzone groups on the `TAZ_North` polygons. The crossing links are the car links whose two end
points lie on different sides of the cordon polygon (a link running along the boundary without
crossing it is not counted); direction by the end point inside.

In [3]:
CORDONS = {
    'Haifa city':            [8, 12, 13, 14, 15, 16, 17],
    'Krayot':                [5, 6, 7],
    'Kiryat Ata + Zevulun':  [9],
    'Nazareth + Nof HaGalil':[19, 20],
    'Tirat Carmel':          [24],
    'Metropolitan core (Haifa, Nesher, Krayot, Kiryat Ata)': [5, 6, 7, 8, 9, 12, 13, 14, 15, 16, 17, 18],
}
net['p0'] = net.geometry.apply(lambda g: Point(g.coords[0])); net['p1'] = net.geometry.apply(lambda g: Point(g.coords[-1]))
p0 = gpd.GeoSeries(net['p0'], crs=CRS); p1 = gpd.GeoSeries(net['p1'], crs=CRS)

def crossing_links(poly):
    in0, in1 = p0.within(poly).values, p1.within(poly).values
    m = net['auto'].values & (in0 != in1)
    x = net.loc[m].copy(); x['direction'] = np.where(in1[m], 'inbound', 'outbound')
    return x

def impute(x):
    """median PCE per lane by road type among the counted crossing links of this cordon; falls back to the network-wide value by type."""
    x = x.copy(); x['lanes'] = x['LANES'].replace(0, 1).fillna(1)
    x['pce_per_lane'] = x['pce_6_9'] / x['lanes']
    med_c = x[x.counted].groupby('TYPE')['pce_per_lane'].median()
    allc = net[net.counted & net.auto].assign(lanes=lambda d: d['LANES'].replace(0, 1).fillna(1))
    med_n = (allc['pce_6_9'] / allc['lanes']).groupby(allc['TYPE']).median()
    x['pce_imputed'] = x['pce_6_9'].astype(float)
    for i in x.index[~x.counted]:
        ty = x.at[i, 'TYPE']; v = med_c.get(ty, np.nan)
        if np.isnan(v): v = med_n.get(ty, med_n.median())
        x.at[i, 'pce_imputed'] = v * x.at[i, 'lanes']
    x['imputed'] = ~x.counted
    for h in ['YARAM6', 'YARAM7', 'YARAM8']:
        x[h + '_imp'] = np.where(x.counted, x[h], x['pce_imputed'] * (x.loc[x.counted, h].sum() / max(x.loc[x.counted, 'pce_6_9'].sum(), 1)))
    return x

import shapely
cent = taz.set_index('TAZ').geometry.representative_point()
cx = cent.x.reindex(TAZ).values; cy = cent.y.reindex(TAZ).values
nz = np.argwhere(car.values > 0); o_idx, d_idx = nz[:, 0], nz[:, 1]
desire = shapely.linestrings(np.stack([np.stack([cx[o_idx], cy[o_idx]], axis=1), np.stack([cx[d_idx], cy[d_idx]], axis=1)], axis=1))
print(f"desire lines for the {len(nz):,} non-empty car cells built")

def through_trips(poly, inside_mask):
    """person trips with both ends outside the cordon whose straight desire line passes through it (car / taxi / bus)."""
    both_out = ~inside_mask[o_idx] & ~inside_mask[d_idx]
    hit = np.zeros(len(nz), bool); hit[both_out] = shapely.intersects(desire[both_out], shapely.prepare(poly) or poly)
    return (car.values[o_idx[hit], d_idx[hit]].sum(), taxi.values[o_idx[hit], d_idx[hit]].sum(), bus.values[o_idx[hit], d_idx[hit]].sum(), int(hit.sum()))

rows, link_rows = [], []
cordon_geoms = {}
for name, szs in CORDONS.items():
    poly = taz[taz.SZ.isin(szs)].geometry.union_all(); cordon_geoms[name] = poly
    inside = set(taz.loc[taz.SZ.isin(szs), 'TAZ']) & set(TAZ); outside = [z for z in TAZ if z not in inside]; inside = sorted(inside)
    inside_mask = np.array([z in set(inside) for z in TAZ])
    T, Ttx, Tbus, n_through = through_trips(poly, inside_mask)
    x = impute(crossing_links(poly)); x['cordon'] = name; link_rows.append(x)
    for d in ['inbound', 'outbound']:
        xd = x[x.direction == d]
        if d == 'inbound': P1, Ptx1, Pbus1 = car.loc[outside, inside].values.sum(), taxi.loc[outside, inside].values.sum(), bus.loc[outside, inside].values.sum()
        else:              P1, Ptx1, Pbus1 = car.loc[inside, outside].values.sum(), taxi.loc[inside, outside].values.sum(), bus.loc[inside, outside].values.sum()
        P, Ptx, Pbus = P1 + T, Ptx1 + Ttx, Pbus1 + Tbus
        veh = P / OCC
        yrs = xd.loc[xd.counted].groupby('count_year')['pce_6_9'].sum(); yrs = (yrs / max(yrs.sum(), 1)).round(2).to_dict()
        pce_c, pce_i = xd.loc[xd.counted, 'pce_6_9'].sum(), xd['pce_imputed'].sum()
        rows.append({'cordon': name, 'direction': d, 'TAZs inside': len(inside), 'crossing car links': len(xd), 'counted': int(xd.counted.sum()),
                     'counted share of imputed PCE': pce_c / pce_i if pce_i else np.nan,
                     'count PCE 06–09 (counted links)': pce_c, 'count PCE 06–09 (imputed total)': pce_i,
                     'count vehicles 06–09 (imputed ÷ PCE factor)': pce_i / PCE,
                     'survey car person trips, one end inside': P1, 'survey car person trips, through (desire line)': T, 'through OD cells': n_through,
                     'survey car person trips': P, 'survey car vehicles (÷ occupancy)': veh,
                     'ratio survey ÷ count': veh / (pce_i / PCE) if pce_i else np.nan,
                     'ratio survey ÷ counted links only (upper bound)': veh / (pce_c / PCE) if pce_c else np.nan,
                     'ratio range (occupancy × PCE factor)': f"{(P/OCC_RANGE[1])/(pce_i/PCE_RANGE[0]):.2f}–{(P/OCC_RANGE[0])/(pce_i/PCE_RANGE[1]):.2f}" if pce_i else '',
                     'taxi-type vehicles (survey)': Ptx / TAXI_OCC, 'bus PCE (survey transit ÷ load × PCE)': Pbus / BUS_LOAD * BUS_PCE,
                     'ratio incl. taxi and bus': (veh + Ptx / TAXI_OCC + Pbus / BUS_LOAD * BUS_PCE / PCE) / (pce_i / PCE) if pce_i else np.nan,
                     'count PHF3h (busiest hour ÷ 3 h, counted links)': xd.loc[xd.counted, ['YARAM6', 'YARAM7', 'YARAM8']].sum().max() / max(pce_c, 1),
                     'count busiest hour': ['06', '07', '08'][int(np.argmax(xd.loc[xd.counted, ['YARAM6', 'YARAM7', 'YARAM8']].sum().values))] + ':00' if pce_c else '',
                     'count year mix (PCE share)': json.dumps({str(int(k)) if k == k else 'n/a': v for k, v in yrs.items()})})
res = pd.DataFrame(rows); links = pd.concat(link_rows)
res.to_csv(f'{OUT}/car_cordon_counts.csv', index=False)
links.drop(columns=['p0', 'p1']).to_csv(f'{OUT}/car_cordon_crossing_links.csv', index=False)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
show = res[['cordon', 'direction', 'crossing car links', 'counted', 'counted share of imputed PCE', 'count PCE 06–09 (counted links)', 'count PCE 06–09 (imputed total)', 'count vehicles 06–09 (imputed ÷ PCE factor)',
            'survey car person trips, one end inside', 'survey car person trips, through (desire line)', 'survey car vehicles (÷ occupancy)', 'ratio survey ÷ count', 'ratio survey ÷ counted links only (upper bound)', 'ratio range (occupancy × PCE factor)', 'ratio incl. taxi and bus']].copy()
show.columns = ['cordon', 'dir', 'links', 'counted', 'counted share', 'PCE counted', 'PCE imputed total', 'count veh', 'survey one-end', 'survey through', 'survey veh', 'ratio', 'ratio vs counted only', 'ratio range', 'ratio incl. taxi+bus']
for c in ['PCE counted', 'PCE imputed total', 'count veh', 'survey one-end', 'survey through', 'survey veh']: show[c] = show[c].round(0)
print(show.round(3).to_string(index=False))
print('\ncount PHF3h on the counted crossing links (busiest clock hour ÷ 06–09) and the busiest hour:'); print(res[['cordon', 'direction', 'count PHF3h (busiest hour ÷ 3 h, counted links)', 'count busiest hour']].round(3).to_string(index=False))
print('\ncount year mix per cordon and direction (PCE share):'); print(res[['cordon', 'direction', 'count year mix (PCE share)']].to_string(index=False))

desire lines for the 24,173 non-empty car cells built


                                               cordon      dir  links  counted  counted share  PCE counted  PCE imputed total  count veh  survey one-end  survey through  survey veh  ratio  ratio vs counted only ratio range  ratio incl. taxi+bus
                                           Haifa city  inbound     39       12          0.317        25191            79539.0    72308.0         65163.0         11995.0     58013.0  0.802                  2.533   0.70–0.93                 0.836
                                           Haifa city outbound     39       12          0.304        22956            75547.0    68679.0         31997.0         11995.0     33076.0  0.482                  1.585   0.42–0.56                 0.491
                                               Krayot  inbound     25        4          0.253        11482            45435.0    41305.0         17128.0         24307.0     31154.0  0.754                  2.985   0.66–0.88                 0.765
                    

## Directional asymmetry and the peak-hour factor

In [4]:
piv = res.pivot(index='cordon', columns='direction', values=['survey car vehicles (÷ occupancy)', 'count vehicles 06–09 (imputed ÷ PCE factor)'])
asym = pd.DataFrame({'survey inbound share': piv[('survey car vehicles (÷ occupancy)', 'inbound')] / (piv[('survey car vehicles (÷ occupancy)', 'inbound')] + piv[('survey car vehicles (÷ occupancy)', 'outbound')]),
                     'count inbound share': piv[('count vehicles 06–09 (imputed ÷ PCE factor)', 'inbound')] / (piv[('count vehicles 06–09 (imputed ÷ PCE factor)', 'inbound')] + piv[('count vehicles 06–09 (imputed ÷ PCE factor)', 'outbound')])})
asym['ratio inbound'] = res[res.direction == 'inbound'].set_index('cordon')['ratio survey ÷ count']; asym['ratio outbound'] = res[res.direction == 'outbound'].set_index('cordon')['ratio survey ÷ count']
print(asym.round(3).to_string())
# the imputation is direction-blind, so the directional test is made on the links counted in BOTH directions only
rev = links.merge(net[['INODE', 'JNODE', 'pce_6_9', 'counted']].rename(columns={'INODE': 'JNODE', 'JNODE': 'INODE', 'pce_6_9': 'pce_rev', 'counted': 'counted_rev'}), on=['INODE', 'JNODE'], how='left')
both = rev[rev.counted & rev.counted_rev.fillna(False)]
pair = both.groupby(['cordon', 'direction'])['pce_6_9'].sum().unstack('direction')
pair['count inbound share (links counted both ways)'] = pair['inbound'] / (pair['inbound'] + pair['outbound']); pair['links counted both ways'] = both.groupby('cordon').size() // 1
pair['survey inbound share'] = asym['survey inbound share']
print('\ndirectional split on the links counted in both directions (PCE 06–09):'); print(pair.round(3).to_string())
asym = asym.join(pair[['count inbound share (links counted both ways)', 'links counted both ways']])
asym.to_csv(f'{OUT}/car_cordon_direction_split.csv')
# hourly profile of the counted crossing links, all cordons pooled, vs the survey's departure-time factor (step 20: car 0.621 study area; corridor by direction 0.661 / 0.626)
phf = pd.read_csv('Output/ths2017/three_mode_2022/peak_hour_factors.csv')
prof = links[links.counted].groupby(['cordon', 'direction'])[HOURS].sum()
prof_share = prof.div(prof.sum(axis=1), axis=0)
am3 = prof[['YARAM6', 'YARAM7', 'YARAM8']]; phf_count = (am3.max(axis=1) / am3.sum(axis=1)).rename('count PHF3h (clock hours)')
pooled = links[links.counted].groupby('cordon')[HOURS].sum(); pooled_share = pooled.div(pooled.sum(axis=1), axis=0)
am_share = pooled[['YARAM6', 'YARAM7', 'YARAM8']].div(pooled[['YARAM6', 'YARAM7', 'YARAM8']].sum(axis=1), axis=0); am_share.columns = ['06:00', '07:00', '08:00']
am_share['busiest clock hour ÷ 3 h'] = am_share.max(axis=1); am_share['× average hour'] = am_share['busiest clock hour ÷ 3 h'] * 3
sv = phf[(phf['level'] == 'study area, all trips') & (phf['layer'] == 'CAR')].iloc[0] if 'level' in phf.columns and ((phf['level'] == 'study area, all trips') & (phf['layer'] == 'CAR')).any() else None
print('\nshares of the 06–09 count in each clock hour on the counted crossing links, per cordon (both directions pooled):'); print(am_share.round(3).to_string())
if sv is not None: print(f"survey car, study area (step 20, departure time): 06 {sv['h6']:.3f} / 07 {sv['h7']:.3f} / 08 {sv['h8']:.3f}; PHF3h (best 60 minutes on 15-minute windows) {sv['PHF3h']:.3f} = {3*sv['PHF3h']:.2f} × an average hour")
print('\nshares of the day (06–19) on the counted crossing links, per cordon:'); print(pooled_share.round(3).to_string())
am_share.to_csv(f'{OUT}/car_cordon_count_am_profile.csv')
print('\ncount PHF3h (busiest clock hour of 06–09 ÷ the three hours; the survey factor is on 15-minute departure windows, 0.621 car study-wide):'); print(phf_count.round(3).to_string())
prof_share.to_csv(f'{OUT}/car_cordon_count_hourly_profile.csv')

                                                       survey inbound share  count inbound share  ratio inbound  ratio outbound
cordon                                                                                                                         
Haifa city                                                            0.637                0.513          0.802           0.482
Kiryat Ata + Zevulun                                                  0.439                0.516          0.658           0.898
Krayot                                                                0.438                0.529          0.754           1.087
Metropolitan core (Haifa, Nesher, Krayot, Kiryat Ata)                 0.564                0.557          0.624           0.607
Nazareth + Nof HaGalil                                                0.391                0.472          0.710           0.988
Tirat Carmel                                                          0.462                0.507        

## The counted links themselves — the largest crossings per cordon

In [5]:
for name in CORDONS:
    x = links[(links.cordon == name)].sort_values('pce_imputed', ascending=False)
    print(f"\n{name}: {len(x)} crossing car links, {x.counted.sum()} counted; largest:")
    print(x[['ID1', 'NAME', 'TYPE', 'LANES', 'direction', 'counted', 'count_year', 'YARAM6', 'YARAM7', 'YARAM8', 'pce_imputed']].head(10).round(0).to_string(index=False))


Haifa city: 78 crossing car links, 24 counted; largest:
        ID1                 NAME  TYPE  LANES direction  counted  count_year  YARAM6  YARAM7  YARAM8  pce_imputed
24370-15133     road 2 /  road 2     1    2.0   inbound     True      2022.0     880    2154    2821       5855.0
12015-49027                  NaN     1    3.0  outbound    False         NaN       0       0       0       4657.0
15372-24370     road 2 /  road 2     1    3.0  outbound     True      2022.0    1203    1653    1801       4657.0
49027-12015                  NaN     1    3.0   inbound    False         NaN       0       0       0       4657.0
15948-15302 ממוצע בין שתי ספירות     2    2.0  outbound     True         NaN     896    1712    1787       4395.0
49033-12215  מחלף דשנים/22 מערבי     2    2.0   inbound     True      2022.0    1131    1424    1591       4146.0
15302-15948 ממוצע בין שתי ספירות     2    2.0   inbound     True         NaN     817    1437    1401       3655.0
16339-16334                  Na

## Figures

In [6]:
fig, ax = plt.subplots(figsize=(9, 10))
taz.boundary.plot(ax=ax, color=GRID, linewidth=0.3)
core = cordon_geoms['Metropolitan core (Haifa, Nesher, Krayot, Kiryat Ata)']
for name, col in zip(list(CORDONS)[:5], [BLUE, ORANGE, AQUA, PURPLE, MUTED]):
    gpd.GeoSeries([cordon_geoms[name]], crs=CRS).boundary.plot(ax=ax, color=col, linewidth=1.6, label=name)
lx = links[links.cordon.isin(list(CORDONS)[:5])]
lx[lx.counted].plot(ax=ax, color=INK, linewidth=1.2, label='crossing link, counted')
lx[~lx.counted].plot(ax=ax, color='#c0392b', linewidth=1.2, linestyle=':', label='crossing link, imputed')
minx, miny, maxx, maxy = core.bounds; pad = 6000
ax.set_xlim(min(minx, cordon_geoms['Nazareth + Nof HaGalil'].bounds[0]) - pad, max(maxx, cordon_geoms['Nazareth + Nof HaGalil'].bounds[2]) + pad); ax.set_ylim(min(miny, cordon_geoms['Tirat Carmel'].bounds[1]) - pad, maxy + pad)
ax.set_axis_off(); ax.legend(loc='lower left', fontsize=8, frameon=False); ax.set_title('Cordons and the network links that cross them (counted / imputed)', fontsize=11, color=INK)
plt.tight_layout(); plt.savefig('Output/figures/car_cordon_map.png', dpi=150); plt.close()

fig, ax = plt.subplots(figsize=(10, 4.5))
SHORT = {'Haifa city': 'Haifa', 'Krayot': 'Krayot', 'Kiryat Ata + Zevulun': 'K. Ata', 'Nazareth + Nof HaGalil': 'Nazareth', 'Tirat Carmel': 'Tirat C.', 'Metropolitan core (Haifa, Nesher, Krayot, Kiryat Ata)': 'Metro core'}
lab = [f"{SHORT[c]}\n{'in' if d == 'inbound' else 'out'}" for c, d in zip(res.cordon, res.direction)]
xs = np.arange(len(res)); ax.bar(xs - 0.2, res['count vehicles 06–09 (imputed ÷ PCE factor)'] / 1000, 0.4, color=ORANGE, label='count (vehicles, imputed total ÷ 1.10)')
ax.bar(xs - 0.2, res['count PCE 06–09 (counted links)'] / PCE / 1000, 0.4, color=INK, alpha=0.25, label='of which on counted links')
ax.bar(xs + 0.2, res['survey car vehicles (÷ occupancy)'] / 1000, 0.4, color=BLUE, label='survey car layer (vehicles, ÷ 1.33)')
for i, r in res.iterrows(): ax.text(i + 0.2, r['survey car vehicles (÷ occupancy)'] / 1000 + 0.3, f"{r['ratio survey ÷ count']:.2f}", ha='center', fontsize=8, color=INK)
ax.set_xticks(xs); ax.set_xticklabels(lab, fontsize=8); ax.set_ylabel('thousand vehicles, 06:00–09:00'); ax.spines[['top', 'right']].set_visible(False); ax.grid(axis='y', color=GRID); ax.set_axisbelow(True)
ax.legend(fontsize=8, frameon=False); ax.set_title('Car layer against the cordon counts (ratio survey ÷ count above each bar)', fontsize=11, color=INK)
plt.tight_layout(); plt.savefig('Output/figures/car_cordon_counts.png', dpi=150); plt.close()
print(f"outputs written to {OUT}/ and Output/figures/car_cordon_*.png  [{time.time()-t0:.0f} s]")

outputs written to Output/validation/ and Output/figures/car_cordon_*.png  [49 s]
